In [3]:
import os
import sys
from pathlib import Path
import re
 
from dotenv import load_dotenv
from tqdm import tqdm
 
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
 
load_dotenv()
 
BASE_DIR    = Path.cwd().parent
BOOKS_DIR   = BASE_DIR / "data" / "raw" / "books"
CHROMA_DIR  = BASE_DIR / "data" / "chroma_db"
COLLECTION  = "babyos_books"
 
CHUNK_SIZE    = 900
CHUNK_OVERLAP = 180

In [4]:
TOPIC_PATTERNS: dict[str, list[str]] = {
    "nutrition": [
        r"\bnutrition\b", r"\bdiet\b", r"\bfolic acid\b", r"\biron\b",
        r"\bvitamin\b", r"\bsupplement\b", r"\bcalcium\b", r"\bomega.3\b",
        r"\bdha\b", r"\bfood\b", r"\beat\b", r"\bcalorie\b", r"\bweight gain\b",
    ],
    "labour": [
        r"\blabour\b", r"\blabor\b", r"\bbirth\b", r"\bdelivery\b",
        r"\bcontractions?\b", r"\bpushing\b", r"\bcrowning\b",
        r"\bepidural\b", r"\bpain relief\b", r"\bcaesarean\b", r"\bc.section\b",
        r"\bwaters? broke?\b", r"\bepisiotomy\b", r"\bforceps\b",
    ],
    "newborn": [
        r"\bnewborn\b", r"\bnappy\b", r"\bdiaper\b", r"\bumbilic\b",
        r"\bjaundice\b", r"\bapgar\b", r"\bfontanelle\b", r"\breflex\b",
        r"\bneonatal\b", r"\binfant\b", r"\bcolostrum\b",
    ],
    "breastfeeding": [
        r"\bbreastfeed\b", r"\bbreast.?feed\b", r"\blatch\b",
        r"\bmilk supply\b", r"\bengorge\b", r"\bnipple\b",
        r"\bformula\b", r"\bwean\b", r"\bbottle.?feed\b",
    ],
    "development": [
        r"\bmilestone\b", r"\bdevelopment\b", r"\bcrawl\b", r"\bwalk\b",
        r"\bspeech\b", r"\blanguage\b", r"\bmotor\b", r"\bcogniti\b",
        r"\btoddler\b", r"\bfirst words?\b", r"\bgrowth chart\b",
    ],
    "mental_health": [
        r"\bdepression\b", r"\banxiety\b", r"\bpnd\b", r"\bpostnatal\b",
        r"\bpostpartum\b", r"\bbaby blues\b", r"\bstress\b",
        r"\bidentity\b", r"\bmatrescence\b", r"\bpatrescence\b",
        r"\bmental health\b", r"\btherapy\b", r"\bcbt\b",
    ],
    "complications": [
        r"\bpreeclampsia\b", r"\bgestational diabetes\b", r"\bpreterm\b",
        r"\bmiscarriage\b", r"\bectopic\b", r"\bplacenta praevia\b",
        r"\biugr\b", r"\bstillbirth\b", r"\bcomplication\b",
        r"\bhypertension\b", r"\bhyperemesis\b",
    ],
    "germany": [
        r"\bmutterpass\b", r"\bhebamme\b", r"\belterngeld\b",
        r"\bvorsorge\b", r"\bkita\b", r"\bkrankenkasse\b",
        r"\bstandesamt\b", r"\bbzga\b", r"\bkindergeld\b",
        r"\bgeburtshaus\b", r"\bkrei.?saal\b",
    ],
}
 
_COMPILED = {
    topic: [re.compile(p, re.IGNORECASE) for p in patterns]
    for topic, patterns in TOPIC_PATTERNS.items()
}
 
 
def tag_chunk(text: str) -> str:
    """Return the single best topic tag for a chunk of text."""
    scores: dict[str, int] = {}
    for topic, patterns in _COMPILED.items():
        score = sum(1 for p in patterns if p.search(text))
        if score:
            scores[topic] = score
    if not scores:
        return "general"
    return max(scores, key=scores.__getitem__)

In [5]:
def _infer_source(filename: str) -> str:
    name = filename.lower()
    if "who"  in name:   return "WHO"
    if "nhs"  in name:   return "NHS"
    if "cdc"  in name:   return "CDC"
    if "acog" in name:   return "ACOG"
    if "aap"  in name:   return "AAP"
    if "bzga" in name or "bundeszentrale" in name: return "BZgA"
    if "unicef" in name: return "UNICEF"
    if "efsa" in name:   return "EFSA"
    if "psi"  in name or "postpartum" in name: return "PSI"
    return "book"
 
 
def _infer_primary_topic(filename: str) -> str:
    """Infer the book's primary topic from its filename for logging."""
    name = filename.lower()
    if any(k in name for k in ["antenatal", "schwanger", "prenatal", "pregnancy"]): return "pregnancy"
    if any(k in name for k in ["labour", "labor", "birth", "geburt"]):   return "labour"
    if any(k in name for k in ["newborn", "neonatal", "postnatal"]):     return "newborn"
    if any(k in name for k in ["stillen", "breastfeed", "lactation"]):   return "breastfeeding"
    if any(k in name for k in ["milestone", "development", "child"]):    return "development"
    if any(k in name for k in ["mental", "depression", "anxiety"]):      return "mental_health"
    if any(k in name for k in ["nutrition", "ernaehrung", "food"]):      return "nutrition"
    if any(k in name for k in ["bzga", "german", "deutsch"]):            return "germany"
    return "general"

In [ ]:
def ingest_books(reset: bool = False) -> None:
    BOOKS_DIR.mkdir(exist_ok=True)
    pdf_files = sorted(BOOKS_DIR.glob("*.pdf"))
 
    if not pdf_files:
        print(f"\n⚠️  No PDFs found in {BOOKS_DIR}")
        print("   See file header for a list of recommended free PDFs to download.")
        print("   Drop them into corpus/books/ then run this script again.")
        return
 
    embeddings   = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=os.getenv("OPENAI_API_KEY"),
    )
    persist_path = str(CHROMA_DIR / COLLECTION)
 
    if reset:
        import shutil
        if Path(persist_path).exists():
            shutil.rmtree(persist_path)
            print(f"🗑️  Wiped '{COLLECTION}'.")
 
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " "],
        length_function=len,
    )
 
    all_chunks: list[Document] = []
    topic_counts: dict[str, int] = {}
    skipped: list[str] = []
 
    for pdf_path in pdf_files:
        fname         = pdf_path.name
        source        = _infer_source(fname)
        primary_topic = _infer_primary_topic(fname)
        print(f"\n📖  {fname}  [{source} / {primary_topic}]")
 
        try:
            pages  = PyPDFLoader(str(pdf_path)).load()
            chunks = splitter.split_documents(pages)
 
            tagged_counts: dict[str, int] = {}
            for i, chunk in enumerate(chunks):
                topic = tag_chunk(chunk.page_content)
                chunk.metadata.update({
                    "source_file":     fname,
                    "source":          source,
                    "primary_topic":   primary_topic,
                    "topic_tag":       topic,          # ← used for filtered MMR
                    "topic_source":    "regex",
                    "collection":      COLLECTION,
                    "chunk_index":     i,
                    "doc_type":        "book_pdf",
                    "total_pages":     len(pages),
                })
                tagged_counts[topic] = tagged_counts.get(topic, 0) + 1
                topic_counts[topic]  = topic_counts.get(topic, 0) + 1
 
            print(f"   {len(pages)} pages → {len(chunks)} chunks")
            print(f"   Topics: { {k: v for k, v in sorted(tagged_counts.items(), key=lambda x: -x[1])} }")
            all_chunks.extend(chunks)
 
        except Exception as e:
            print(f"   ❌  Failed: {e}")
            skipped.append(fname)
 
    if not all_chunks:
        print("\nNo chunks to index.")
        return
 
    print(f"\n📥  Indexing {len(all_chunks)} chunks into '{COLLECTION}'...")
 
    batch_size  = 100
    vectorstore = None
 
    for i in tqdm(range(0, len(all_chunks), batch_size)):
        batch = all_chunks[i: i + batch_size]
        if vectorstore is None:
            vectorstore = Chroma.from_documents(
                documents=batch,
                embedding=embeddings,
                collection_name=COLLECTION,
                persist_directory=persist_path,
            )
        else:
            vectorstore.add_documents(batch)
            
    print(f"\n✅  '{COLLECTION}' complete — {len(all_chunks)} chunks")
    print("\n   Topic distribution:")
    for topic, count in sorted(topic_counts.items(), key=lambda x: -x[1]):
        bar = "█" * min(count // 5, 40)
        print(f"   {topic:<16} {count:>5}  {bar}")
 
    if skipped:
        print(f"\n⚠️  Skipped: {', '.join(skipped)}")


In [7]:
import argparse
parser = argparse.ArgumentParser(description="BabyOS PDF Book Ingestion v2")
parser.add_argument("--reset", action="store_true", help="Wipe and rebuild")
args, unknown = parser.parse_known_args()
ingest_books(reset=args.reset)


📖  Eng.-12-months-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 8 chunks
   Topics: {'general': 6, 'development': 2}

📖  Eng.-15-mo.-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 9 chunks
   Topics: {'general': 5, 'development': 4}

📖  Eng.-18-mo-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 9 chunks
   Topics: {'general': 7, 'development': 2}

📖  Eng.-2-months-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 7 chunks
   Topics: {'general': 5, 'development': 2}

📖  Eng.-2-yrs-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 8 chunks
   Topics: {'general': 6, 'development': 2}

📖  Eng.-30-mo.-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 9 chunks
   Topics: {'nutrition': 5, 'development': 3, 'general': 1}

📖  Eng.-4-months-Milestone-Moments-Checklist-2021-P.pdf  [book / development]
   2 pages → 7 chunks
   Topics: {'general':

100%|██████████| 25/25 [00:23<00:00,  1.08it/s]


✅  'babyos_books' complete — 2496 chunks

   Topic distribution:
   labour             809  ████████████████████████████████████████
   general            788  ████████████████████████████████████████
   nutrition          296  ████████████████████████████████████████
   newborn            255  ████████████████████████████████████████
   mental_health      156  ███████████████████████████████
   development         77  ███████████████
   complications       72  ██████████████
   breastfeeding       43  ████████
